In [58]:
import importlib
import tiktoken
import torch

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [59]:
tokenizer = tiktoken.get_encoding("cl100k_base")



text = (
    "Malavrba mrda<|endoftext|>"
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

strings = tokenizer.decode(integers)
print(strings)



[30700, 70684, 4749, 17767, 3315, 100257]
Malavrba mrda<|endoftext|>


# Question 2 

Encode Shakespeare entire's text and be able to decode it as well.

In [60]:
with open('t8.shakespeare.txt', 'r', encoding='utf-8') as file:
    shakespeare_vocab = file.read()

In [61]:
# Some Extract for testing

shakespeare_text = """From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.<|endoftext|>When forty winters shall besiege thy brow,
  And dig deep trenches in thy beauty's field,
  Thy youth's proud livery so gazed on now,
  Will be a tattered weed of small worth held:  
  Then being asked, where all thy beauty lies,
  Where all the treasure of thy lusty days;
  To say within thine own deep sunken eyes,
  Were an all-eating shame, and thriftless praise.
  How much more praise deserved thy beauty's use,
  If thou couldst answer 'This fair child of mine
  Shall sum my count, and make my old excuse'
  Proving his beauty by succession thine.
    This were to be new made when thou art old,
    And see thy blood warm when thou feel'st it cold"""

In [62]:
encoded_integer = tokenizer.encode(shakespeare_vocab, allowed_special={"<|endoftext|>"})

vocab_size = len(encoded_integer)
print(vocab_size)

1472163


In [63]:
encoded_integer=tokenizer.encode(shakespeare_text, allowed_special={"<|endoftext|>"})

print(encoded_integer)



[3915, 20028, 267, 20566, 584, 12876, 5376, 345, 220, 3011, 28592, 13444, 596, 16392, 2643, 2646, 2815, 345, 220, 2030, 439, 279, 436, 13154, 1288, 555, 892, 31952, 521, 345, 220, 5414, 28682, 51543, 2643, 11984, 813, 5044, 512, 220, 2030, 34223, 51068, 311, 270, 483, 1866, 10107, 6548, 345, 220, 29970, 596, 83, 26236, 3177, 596, 35678, 449, 659, 18451, 77057, 10633, 345, 220, 25274, 264, 79054, 1405, 37492, 15812, 345, 220, 67675, 659, 26236, 53077, 11, 311, 26236, 10437, 659, 2288, 28128, 512, 220, 86471, 430, 1989, 1457, 279, 1917, 596, 7878, 79760, 345, 220, 1628, 1193, 65206, 311, 279, 342, 8039, 88, 10683, 345, 220, 25218, 270, 483, 1866, 37808, 293, 6198, 478, 26236, 2262, 345, 220, 1628, 28682, 523, 1103, 52016, 596, 83, 12571, 304, 308, 20831, 29510, 512, 262, 393, 488, 279, 1917, 11, 477, 775, 420, 2840, 973, 387, 345, 262, 2057, 8343, 279, 1917, 596, 4245, 11, 555, 279, 25165, 323, 40344, 13, 100257, 4599, 36498, 86082, 4985, 92728, 713, 26236, 60375, 345, 220, 1628, 4170, 5

In [64]:
decoded_string = tokenizer.decode(encoded_integer)
print(decoded_string)

From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.<|endoftext|>When forty winters shall besiege thy brow,
  And dig deep trenches in thy beauty's field,
  Thy youth's proud livery so gazed on now,
  Will be a tattered weed of small worth held:  
  Then being asked, where all thy beauty lies,
  Where all the treasure of thy lusty days;
  To say within thine own deep sunken eyes,
  Were an all-eating shame, an

# Question 3

Work with custom Data loader

In [65]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
    
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("cl100k_base")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

dataloader = create_dataloader_v1(shakespeare_vocab, batch_size=4, max_length=6, stride=6)
for batch in dataloader:
    input_ids, target_ids = batch
    print("Input IDs shape:", input_ids.shape)
    print("Target IDs shape:", target_ids.shape)
    break

print("Input IDs:", input_ids)

Input IDs shape: torch.Size([4, 6])
Target IDs shape: torch.Size([4, 6])
Input IDs: tensor([[25753,   280,   262,  5414, 69540,   656],
        [ 1372,    13,   358,   656, 63278,   757],
        [41630,    11, 51451,    11, 19781,  4950],
        [   11,   856, 38031,   627,   262, 46332]])


In [66]:

example = tokenizer.encode(shakespeare_text, allowed_special={"<|endoftext|>"})  # encode first
context_size = 4
for i in range(1, context_size+1):
    input = example[:i]
    target = example[i]
    print(tokenizer.decode(input), "---->", tokenizer.decode([target]))

From ---->  faire
From faire ----> st
From fairest ---->  creatures
From fairest creatures ---->  we


# Question 4

In [67]:
%run transformer_walkthrough.py --case 1

step 0: train loss 4.2000, val loss 4.2047
step 500: train loss 2.6911, val loss 2.7087
step 1000: train loss 2.5196, val loss 2.5303
step 1500: train loss 2.4775, val loss 2.4829
step 2000: train loss 2.4408, val loss 2.4523
step 2500: train loss 2.4272, val loss 2.4435
step 3000: train loss 2.4130, val loss 2.4327
step 3500: train loss 2.3956, val loss 2.4212
step 4000: train loss 2.4041, val loss 2.3992
step 4500: train loss 2.3980, val loss 2.4084

K:
NGey

Letnrad wineam:
Kicou hitipteavimancraby whet muthe hus darge.

Wind!
IRD: Ind, tind spoof om and f.
Sy stllalevere here me honouen fot in,
So and, vist orby?
Thar hous mat deest she rd?

Wowin wof t, ath th ay miligiryouchth-orto mou tenges, ald pors banebe y prothetack aklel I veriplansnidierd avit for,
KI thit ndist allll perd the:
Acu Empoouthant, I to
Ten mar.

S:
Bugh the I hy nd meis moh h!


AThamen es ty I has.

MI ithe thensterat blo gaar,
A d muts ed ronur wiend tl-ou,
Therim


In [68]:
%run transformer_walkthrough.py --case 6

step 0: train loss 4.5196, val loss 4.5140
step 500: train loss 2.2801, val loss 2.2913
step 1000: train loss 2.1422, val loss 2.1700
step 1500: train loss 2.0547, val loss 2.1089
step 2000: train loss 2.0389, val loss 2.1026
step 2500: train loss 2.0021, val loss 2.0721
step 3000: train loss 1.9706, val loss 2.0565
step 3500: train loss 1.9246, val loss 2.0311
step 4000: train loss 1.9222, val loss 2.0181
step 4500: train loss 1.9112, val loss 2.0256

And bith
Rome the dlet son
morld in that saink,
Nor yout grabalt!

Ghan, and ret our him,
Het. wars the wardice
thou thee our look inst not armal.

NORIOLINAUS:
Cicchast,
Where love Rispors,
Pather ;out thus her sorn.

IXFortchess? 
ULIt where can pitch fulr hird schoming 'Foul thour set duke
Stight to carth if Goot wents, leven,
Andnow, be rewet say mroke upo, parity ling'st bed dief, be refuct, justist limink, beimes to dendear you cany ston the which,
The dult,
Thome,
To, by. give who.

DUKE 


## Explanation 

Case 1 uses the Single-head attention training algorithm while case 6 uses Dropout.

Where they both don't seem like english, I think 6 is slightly more improved, provides a little better structure and a little better wording. I counted more english letters.